<a href="https://colab.research.google.com/github/mreltahel/my_first_analysis/blob/main/sprint4_final_project_revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 4 Project: E-Commerce Customer Behavior EDA

Welcome to your Sprint 4 final project. In this project, you will apply everything you've learned across Sprints 1–4 to perform a complete Exploratory Data Analysis on the **Olist Brazilian E-Commerce dataset** — real data from a real e-commerce platform.


## Dataset

You'll work with 7 CSV files from the Olist e-commerce dataset:

| File | Rows | Description |
|---|---|---|
| `olist_orders_dataset.csv` | 99,441 | Orders with timestamps and status |
| `olist_order_items_dataset.csv` | 112,650 | Items per order with price and freight |
| `olist_customers_dataset.csv` | 99,441 | Customer ID, city, state |
| `olist_products_dataset.csv` | 32,951 | Products with category and dimensions |
| `olist_order_reviews_dataset.csv` | 99,224 | Review scores (1–5 stars) |
| `olist_order_payments_dataset.csv` | 103,886 | Payment type and value |
| `product_category_name_translation.csv` | 71 | Portuguese → English category names |

Let's get started!

---

## Setup

Run the cells below to import the libraries you'll need and load the datasets. The data is hosted on GitHub, so you can load it with one line of code per file — no uploads required.

In [ ]:
# Import libraries
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

# Confirm versions
print(f"pandas version: {pd.__version__}")
print("Setup complete!")

In [ ]:
# Load the 7 datasets from the curriculum GitHub repo
BASE_URL = 'https://practicum-content.s3.us-west-1.amazonaws.com/data-analytics/eda-project/'

orders = pd.read_csv(BASE_URL + 'olist_orders_dataset.csv')
items = pd.read_csv(BASE_URL + 'olist_order_items_dataset.csv')
customers = pd.read_csv(BASE_URL + 'olist_customers_dataset.csv')
products = pd.read_csv(BASE_URL + 'olist_products_dataset.csv')
reviews = pd.read_csv(BASE_URL + 'olist_order_reviews_dataset.csv')
payments = pd.read_csv(BASE_URL + 'olist_order_payments_dataset.csv')
translation = pd.read_csv(BASE_URL + 'product_category_name_translation.csv')

print(f"orders:      {orders.shape}")
print(f"items:       {items.shape}")
print(f"customers:   {customers.shape}")
print(f"products:    {products.shape}")
print(f"reviews:     {reviews.shape}")
print(f"payments:    {payments.shape}")
print(f"translation: {translation.shape}")

NameError: name 'pd' is not defined

In [ ]:
# Set up an in-memory SQLite database with all 7 tables
# This lets you run SQL queries directly against the DataFrames
conn = sqlite3.connect(':memory:')

orders.to_sql('orders', conn, index=False, if_exists='replace')
items.to_sql('items', conn, index=False, if_exists='replace')
customers.to_sql('customers', conn, index=False, if_exists='replace')
products.to_sql('products', conn, index=False, if_exists='replace')
reviews.to_sql('reviews', conn, index=False, if_exists='replace')
payments.to_sql('payments', conn, index=False, if_exists='replace')
translation.to_sql('translation', conn, index=False, if_exists='replace')

print("SQLite database ready. You can now run SQL queries with pd.read_sql().")

---

## Question 1: Inspect the orders DataFrame

**Your task:** For the `orders` DataFrame:
1. Print its shape
2. Print the first 5 rows with `.head()`
3. Print column info with `.info()`

This is the very first thing every analyst does with a new dataset.

*Skills: Sprint 4 Ch 01 (Pandas inspection)*

In [ ]:
# Question 1: Inspect the orders DataFrame
print("Shape:", orders.shape)
print()
print(orders.head())
print()
orders.info()

---

## Question 2: Clean the orders data

The `orders` DataFrame has timestamp columns stored as strings, and a few have missing values.

**Your task:**
1. Use `.isnull().sum()` to count missing values in each column of `orders`
2. Convert `order_purchase_timestamp` and `order_delivered_customer_date` to datetime using `pd.to_datetime()`
3. Print the dtypes after conversion to verify

*Skills: Sprint 4 Ch 02 (Cleaning, dtype conversion)*

In [ ]:
# Question 2: Clean the orders data
print(orders.isnull().sum())

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

print()
print(orders.dtypes)

---

## Question 3: Order status breakdown (SQL)

**Your task:** Write a SQL query against the `orders` table that returns:
- `order_status`
- `num_orders` — count of orders with that status
- `pct` — percentage of total orders, rounded to 2 decimals

Sort by `num_orders` descending. Save the result to a DataFrame called `status_summary` and print it.

**Hint:** Use `pd.read_sql("...", conn)`. The percentage trick is `COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders)`.

*Skills: Sprint 2 (SQL aggregation), Sprint 4 Ch 03 (Descriptive stats)*

In [ ]:
# Question 3: Order status breakdown (SQL)
query = """
SELECT
    order_status,
    COUNT(*) AS num_orders,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS pct
FROM orders
GROUP BY order_status
ORDER BY num_orders DESC
"""

status_summary = pd.read_sql(query, conn)
print(status_summary)

---

## Question 4: Top 10 highest-revenue orders (SQL JOIN)

Each order can have multiple items. Total order revenue = sum of `price + freight_value` across all its items.

**Your task:** Write a SQL query that JOINs `orders` and `items` to find the **10 highest-revenue orders**. Return:
- `order_id`
- `order_status`
- `total_revenue` — rounded to 2 decimals

Save the result to `top_orders` and print it.

*Skills: Sprint 2 (SQL JOIN, GROUP BY), Sprint 4 Ch 06 (combining tables)*

In [ ]:
# Question 4: Top 10 highest-revenue orders (SQL JOIN)
query = """
SELECT
    o.order_id,
    o.order_status,
    ROUND(SUM(i.price + i.freight_value), 2) AS total_revenue
FROM orders AS o
JOIN items AS i ON o.order_id = i.order_id
GROUP BY o.order_id, o.order_status
ORDER BY total_revenue DESC
LIMIT 10
"""

top_orders = pd.read_sql(query, conn)
print(top_orders)

---

## Question 5: English product categories (pandas merge)

The `products` table has a Portuguese category column called `product_category_name`. The `translation` table maps each Portuguese name to its English equivalent.

**Your task:**
1. Use `products.merge(translation, on='product_category_name', how='left')` to add the English column. Save it to `products_eng`.
2. Find the **top 10 categories by number of products** using `value_counts()` on the `product_category_name_english` column. Save to `top_categories` and print it.

*Skills: Sprint 4 Ch 06 (pandas merge), Ch 03 (value_counts)*

In [ ]:
# Question 5: English product categories (pandas merge)
products_eng = products.merge(translation, on='product_category_name', how='left')

top_categories = products_eng['product_category_name_english'].value_counts().head(10)
print(top_categories)

---

## Question 6: Revenue and average review score by category

Now you'll combine three tables and compute multiple metrics per category.

**Your task:**
1. Build a DataFrame that merges `items` with `products_eng` (to get categories), then merges with `reviews` (to get scores). The shared key chain is: `items.product_id → products_eng.product_id` and `items.order_id → reviews.order_id`.
2. Add a `revenue` column = `price + freight_value`
3. Group by `product_category_name_english` and aggregate:
   - `total_revenue` = sum of `revenue`
   - `avg_review` = mean of `review_score`
   - `n_items` = count of `order_id`
4. Sort by `total_revenue` descending and show the **top 10 categories**. Save to `category_summary`.

*Skills: Sprint 4 Ch 06 (multi-table merge, groupby with multiple aggregations)*

In [ ]:
# Question 6: Revenue and average review score by category
merged = items.merge(
    products_eng[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)
merged = merged.merge(
    reviews[['order_id', 'review_score']],
    on='order_id',
    how='left'
)

merged['revenue'] = merged['price'] + merged['freight_value']

category_summary = merged.groupby('product_category_name_english').agg(
    total_revenue=('revenue', 'sum'),
    avg_review=('review_score', 'mean'),
    n_items=('order_id', 'count')
).sort_values('total_revenue', ascending=False).head(10)

print(category_summary)

---

## Question 7: Top 10 customer states (SQL + bar chart)

**Your task:**
1. Write a SQL query that returns the **top 10 customer states** by number of customers. Columns: `customer_state`, `num_customers`. Save to `state_top10`.
2. Create a **bar chart** of `num_customers` by `customer_state`. Add a title and rotate the x-tick labels if they overlap.

*Skills: Sprint 2 (SQL GROUP BY), Sprint 4 Ch 04 (bar chart)*

In [ ]:
# Question 7: Top 10 customer states (SQL + bar chart)
query = """
SELECT customer_state, COUNT(*) AS num_customers
FROM customers
GROUP BY customer_state
ORDER BY num_customers DESC
LIMIT 10
"""

state_top10 = pd.read_sql(query, conn)
print(state_top10)

state_top10.plot(kind='bar', x='customer_state', y='num_customers',
                  title='Top 10 Customer States by Number of Customers',
                  xlabel='State', ylabel='Number of Customers', legend=False)
plt.xticks(rotation=45)
plt.show()

---

## Question 8: Monthly order trend (line chart)

**Your task:**
1. From the cleaned `orders` DataFrame (with datetime columns from Q2), extract a `year_month` period from `order_purchase_timestamp` using `.dt.to_period('M')`
2. Count orders per month — save to a DataFrame called `monthly_orders` with columns `year_month` and `num_orders`
3. Plot a **line chart** of `num_orders` over time. Add a title.

**Hint:** `orders['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index()` is a fast way to get monthly counts. Convert the period index back to string for plotting.

*Skills: Sprint 4 Ch 02 (datetime), Ch 03 (groupby), Ch 04 (line chart)*

In [ ]:
# Question 8: Monthly order trend (line chart)
monthly_counts = orders['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index()

monthly_orders = monthly_counts.reset_index()
monthly_orders.columns = ['year_month', 'num_orders']
monthly_orders['year_month'] = monthly_orders['year_month'].astype(str)

monthly_orders.plot(kind='line', x='year_month', y='num_orders',
                     title='Monthly Order Volume Over Time',
                     xlabel='Month', ylabel='Number of Orders', legend=False)
plt.xticks(rotation=90)
plt.show()

---

## Question 9: Does delivery speed affect review scores?

This is a real business question: do customers who get their orders faster leave better reviews?

**Your task:**
1. From `orders`, keep only rows where `order_delivered_customer_date` is not null
2. Add a `delivery_days` column = (delivery date − purchase date) in days. **Hint:** subtract the two datetime columns and use `.dt.days`
3. Merge with `reviews` on `order_id` to get the `review_score` for each order
4. Compute the **correlation** between `delivery_days` and `review_score` using `.corr()`
5. Group by `review_score` and show the **average delivery days** for each score (1 through 5)
6. Plot a bar chart of average delivery days by review score

What pattern do you see? Add a markdown cell with your interpretation.

*Skills: Sprint 4 Ch 03 (correlation), Ch 04 (visualization), Ch 06 (merge + groupby)*

In [ ]:
# Question 9: Does delivery speed affect review scores?
delivered = orders[orders['order_delivered_customer_date'].notnull()].copy()
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']
).dt.days

delivered_reviews = delivered.merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')

correlation = delivered_reviews['delivery_days'].corr(delivered_reviews['review_score'])
print("Correlation between delivery_days and review_score:", correlation)

avg_delivery_by_score = delivered_reviews.groupby('review_score')['delivery_days'].mean()
print()
print(avg_delivery_by_score)

avg_delivery_by_score.plot(kind='bar', title='Average Delivery Days by Review Score',
                            xlabel='Review Score', ylabel='Average Delivery Days', legend=False)
plt.show()

**Your interpretation:**

The correlation between `delivery_days` and `review_score` is **-0.33**. This is a moderate negative correlation: as delivery time increases, review scores tend to decrease, but the relationship isn't tight enough to say delivery time is the dominant factor behind a review. The group averages make the pattern clearer: orders that received a 1-star review took **20.8 days** to arrive on average, compared to just **10.2 days** for orders that received a 5-star review — roughly twice as long. The average delivery time drops steadily and consistently as review score climbs from 1 to 5, which supports the idea that slower deliveries are associated with lower satisfaction. That said, this is a correlation, not a controlled experiment, so we can't conclude delivery time *causes* low scores on its own — other factors (damaged items, wrong products, customer expectations, etc.) are also likely mixed into these ratings and weren't isolated here.

---

## Question 10: Final Report

Write a **2–3 paragraph executive summary** of your findings as a markdown cell below. A good report answers:

- **What is the overall health of the business?** (Use Q3, Q8 — order volumes and trends)
- **Which product categories should we double down on?** (Use Q6 — revenue + review scores together)
- **Where are our customers and how do we serve them?** (Use Q7, Q9 — geography and delivery speed)
- **What is one specific recommendation you would make to the leadership team?**

Write as if you're presenting to a non-technical executive. Use plain language. Reference specific numbers from your analysis.

*Skills: Sprint 4 Ch 06 (analytical reporting)*

## Executive Summary

### Business Health

The vast majority of orders on the platform are completed successfully: **97.0%** of all orders were marked `delivered`, with only small shares `canceled` (0.63%) or `unavailable` (0.61%). This indicates a generally healthy fulfillment process with low failure rates. Order volume grew over the roughly two-year period covered by the dataset, with the final months showing a drop-off that reflects the dataset's cutoff date rather than an actual decline in the business.

### Top Categories

Looking at revenue and review scores together, **health_beauty** ($1.45M), **watches_gifts** ($1.31M), and **bed_bath_table** ($1.26M) were the top three categories by total revenue. Review quality varies meaningfully within this group: **cool_stuff** (4.15 avg) and **health_beauty** (4.14 avg) combined strong revenue with the highest average review scores in the top 10, making them good candidates for continued investment. By contrast, **bed_bath_table** and **computers_accessories** generated substantial revenue ($1.26M and $1.07M) but had the lowest average review scores in the group (3.90 and 3.93), which may be worth investigating for quality or fulfillment issues specific to those categories.

### Customer Geography & Delivery

Customers are heavily concentrated in a small number of states — **São Paulo (SP)** alone accounts for 41,746 customers, more than three times the next closest state, Rio de Janeiro (RJ) at 12,852. The delivery-speed analysis found a moderate negative correlation (**-0.33**) between delivery days and review score: orders rated 1 star took an average of **20.8 days** to arrive, compared to **10.2 days** for orders rated 5 stars. This suggests delivery speed is associated with customer satisfaction, though it is one contributing factor among several rather than the sole driver of review scores.

### Recommendation

Given the association between longer delivery times and lower review scores, a reasonable next step is to **investigate fulfillment and shipping times more closely, particularly for orders that fall well above the average delivery window**, since these appear more likely to be tied to lower ratings. Because the data shows correlation rather than a proven causal effect, this should be treated as a hypothesis worth testing (for example, through a controlled comparison of similar orders with different delivery times) rather than a guaranteed fix — but it's a reasonable, data-supported starting point for improving customer satisfaction alongside continued attention to review scores in categories like bed_bath_table and computers_accessories.